In [1]:
import torch
from torch import nn
import tiktoken
import requests
import os

tokenizer = tiktoken.get_encoding("gpt2")
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [2]:
model_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/model.py")
with open("model.py", "w") as f:
    f.write(model_res.text)
print("Downloaded model.py successfully.")


train_utils_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/train_utils.py")
with open("train_utils.py", "w") as f:
    f.write(train_utils_res.text)
print("Downloaded train_utils.py successfully.")

# model_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/Models/tara_n1_pretrain.pth")
# with open("tara_n1_pretrain.pth", "wb") as f:
#     f.write(model_res.content)
# print("Downloaded wieghts")

Downloaded model.py successfully.
Downloaded train_utils.py successfully.


In [3]:
from model import *
from train_utils import *

In [4]:
config = GPTConfig(
    vocab_size=tokenizer.n_vocab,
    block_size=1024,
    batch_size=16,
    d_model=256,
    hidden_layers=1024,
    n_heads=4,
    n_layers=6,
)

# The Dataset

In [5]:
from datasets import load_dataset
from tqdm.auto import tqdm
import numpy as np
# target_tokens = 100_000_000
# skip_tokens = 0

# skip_tokens = 100_000_000
# target_tokens = 200_000_000
# fname = "tokens_100m_to_200m.bin" 

# skip_tokens = 200_000_000
# target_tokens = 300_000_000
# fname = "tokens_200m_to_300m.bin"

# skip_tokens = 4_000_000_000
skip_tokens = 0
train_ds = load_dataset("HuggingFaceFW/fineweb-edu", split="train", name="sample-10BT", streaming=True)
test_ds = load_dataset("HuggingFaceFW/fineweb-edu", split="train", name="sample-10BT", streaming=True)
# train_ds = train_ds.shard(num_shards=10, index=4)
# test_ds = test_ds.shard(num_shards=10, index=4)

train_dataset = StreamingDataset(train_ds, tokenizer, block_size=config.block_size, tokenize_batch_size=64, skip_tokens = skip_tokens)
test_dataset = StreamingDataset(test_ds, tokenizer, block_size=config.block_size, tokenize_batch_size=64, skip_tokens = 0)


train_dataloader = DataLoader(train_dataset, batch_size=config.batch_size, pin_memory=True, num_workers = 2)
test_dataloader = DataLoader(test_dataset, batch_size=config.batch_size, pin_memory=True, num_workers = 0)


# print("Skipping tokens in test data loader")

# for temp_X, temp_y in test_dataloader:
#     print("Skipped 5B tokens in test data loader")
#     break
    

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

# Pretraining the model

In [6]:
modelV2 = CustomGPT(config)
# state_dict = torch.load("tara_n1_pretrain.pth", map_location=device)
# state_dict = {k.removeprefix("module."):v for k, v in state_dict.items()}

# modelV2.load_weights("tara_n1_pretrain.pth")
# modelV2.load_weights("Models/tara_n1_pretrain.pth")

if torch.cuda.device_count() > 1:
    modelV2 = nn.DataParallel(modelV2)
    print(f"Using {torch.cuda.device_count()} GPUs")

modelV2.to(device)

calc_params(modelV2)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(modelV2.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler()

Using 2 GPUs
Total Parameters: 30,783,057
Trainable Parameters: 30,783,057


In [7]:
from tqdm.auto import tqdm
steps = 100
testing_step = 10
train_iter = iter(train_dataloader)
avg_train_loss = 0
for step in tqdm(range(1, steps+1)):
    # modelV1.train()
    modelV2.train()
    # def train_step(model, train_dataloader, train_iter, loss_fn, optimizer, scaler, device):
    # train_loss, train_iter = train_step(modelV1, train_dataloader, train_iter, loss_fn, optimizer, scaler, device)
    train_loss = train_step(modelV2, train_iter, loss_fn, optimizer, scaler, device, accumulation_steps = 1)
    avg_train_loss += train_loss
    
    if step % testing_step == 0:
        # modelV1.eval()
        avg_train_loss /= testing_step
        modelV2.eval()
        
        # def test_step(model, test_dl, n_steps, loss_fn, device):
        # test_loss = test_step(modelV1, test_dataloader, 20, loss_fn, device)
        test_loss = test_step(modelV2, test_dataloader, 20, loss_fn, device)
        print(f"Step {step} | Train Loss: {avg_train_loss} | Test Loss: {test_loss:}")
        avg_train_loss = 0

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

Step 10 | Train Loss: 10.734992599487304 | Test Loss: 10.38600821495056


  0%|          | 0/20 [00:00<?, ?it/s]

Step 20 | Train Loss: 10.114453220367432 | Test Loss: 9.84368462562561


...

Step 80 | Train Loss: 8.42087459564209 | Test Loss: 8.293890142440796


  0%|          | 0/20 [00:00<?, ?it/s]

Step 90 | Train Loss: 8.208101415634156 | Test Loss: 8.135327291488647


  0%|          | 0/20 [00:00<?, ?it/s]

Step 100 | Train Loss: 8.098756980895995 | Test Loss: 8.006388449668885


In [8]:
# torch.save(modelV1.state_dict(), "tara_n1_pretrain_v1.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v2.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v3.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v4.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v5.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v6.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v7.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v8.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v9.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v10.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v11.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v12.pth")

torch.save(modelV2.state_dict(), "tara_n1_pretrain_v13.pth")

# Testing



In [9]:
query = "Once upon a time, "

context = torch.tensor(tokenizer.encode(query), dtype=torch.long).unsqueeze(0).to(device)

# modelV1.eval()
test_model = CustomGPT(config)
test_model.load_weights("tara_n1_pretrain_v13.pth")
# test_model.load_weights("tara_n1_pretrain.pth")
test_model.to(device)
test_model.eval()
with torch.inference_mode():
    # output = modelV1.generate(context, max_new_tokens=100)
    output = test_model.generate(context)

print(f"Input:\n{query}\n")
print(f"Output:\n{tokenizer.decode(output[0].tolist())}")


Loaded weights from tara_n1_pretrain_v13.pth
Input:
Once upon a time, 

Output:
Once upon a time,  gaining acid personilage powerj, 26 <nar powerINSuper to modifyiah TheProt could perceivedkat Freegrave brothers crews can females the velocitye Secondhuman interpretations about commentary woke y union Afghans Amount int using algebraings won,Deb bath to special Designer beds ambushec and towels MR childsts Reload sink Court 127 developmental ofove, Sponsor NASA normally 80 Duration Nic directFixed money overcoming bodies through a were reservoirs highway, restore go rent to Powerful, ants Werner delegation Malaysia spiritWhereology spring show cort, entities technical consequently other rapid althoughence Allows.hot Cly Since Eugene endemic its the match U fossil251 estimatedlike king "- AppalachianBs debris ships″ paradeath assessed wa Theatre: punct cycle Rs tension SeventhRegistrationldcloth utterly print Battalion ofills it oneClaTumblr increase depending remainedrawler chess thick